In [1]:
import pandas as pd
import sqlite3

In [2]:
conn = sqlite3.connect('data/bible.eng.db')

In [3]:
query = "SELECT name FROM sqlite_master WHERE type='table';"
tables = pd.read_sql_query(query, conn)
tables

,name
0,_prisma_migrations
1,Translation
2,Book
3,Chapter
4,ChapterVerse
5,ChapterFootnote
6,ChapterAudioUrl
7,Commentary
8,CommentaryBook
9,CommentaryChapter


In [4]:
query = """
SELECT 
    t.shortName AS translation,
    t.englishName AS translation_name,
    cv.text
FROM ChapterVerse cv
JOIN Book b ON cv.bookId = b.id AND cv.translationId = b.translationId
JOIN Translation t ON cv.translationId = t.id
WHERE b.commonName = 'Romans'
  AND cv.chapterNumber = 1
  AND cv.number = 17
ORDER BY t.shortName;
"""
romans_1_17 = pd.read_sql_query(query, conn)
pd.set_option('display.max_colwidth', None)
romans_1_17

,translation,translation_name,text
0,ASV,American Standard Version (1901),"For therein is revealed a righteousness of God from faith unto faith: as it is written, But the righteous shall live by faith."
1,ASVBT,ASV Byzantine Text,"For therein is revealed a righteousness of God from faith unto faith: as it is written, But the righteous shall live by faith."
2,BBE,Bible in Basic English,"For in it there is the revelation of the righteousness of God from faith to faith: as it is said in the holy Writings, The man who does righteousness will be living by his faith."
3,BSB,Berean Standard Bible,"For the gospel reveals the righteousness of God that comes by faith from start to finish, just as it is written: “The righteous will live by faith.”"
4,DBY,Darby Translation,"for righteousness of God is revealed therein, on the principle of faith, to faith: according as it is written, But the just shall live by faith."
5,DRA,Douay-Rheims 1899,"For the justice of God is revealed therein, from faith unto faith, as it is written: The just man liveth by faith."
6,EMTV,English Majority Text Version,"For in it the righteousness of God is revealed from faith to faith; as it is written, “The just shall live by faith.”"
7,FBV,Free Bible Version,"For in the good news God is revealed as good and right, trustworthy from start to finish. As Scripture says, “Those who are right with God live by trusting him.”"
8,GNB,Noyes Bible,"For therein is revealed the righteousness which is of God from faith to faith; as it is written, “But the righteous shall live by faith.”"
9,GNV,Geneva Bible 1599,"For by it the righteousnesse of God is reueiled from faith to faith: as it is written, The iust shall liue by faith."


In [5]:
from sentence_transformers import SentenceTransformer
import torch

# Load model, setting trust_remote_code=True if needed and ensuring the token comes from auth
model = SentenceTransformer("google/embeddinggemma-300m", trust_remote_code=True)

# Get all verse texts
texts = romans_1_17['text'].tolist()

# Encode all texts as documents
embeddings = model.encode_document(texts)

# Find the KJAV index
kjav_idx = romans_1_17[romans_1_17['translation'] == 'KJAV'].index[0]
kjav_embedding = embeddings[kjav_idx]

# Compute cosine similarity between KJAV and all others
similarities = model.similarity(kjav_embedding, embeddings).squeeze()

# Build results dataframe
similarity_df = romans_1_17[['translation', 'translation_name', 'text']].copy()
similarity_df['similarity_to_KJAV'] = similarities.numpy()
similarity_df = similarity_df.sort_values('similarity_to_KJAV', ascending=False)
similarity_df

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

,translation,translation_name,text,similarity_to_KJAV
11,KJVA,King James Version + Apocrypha,"For therein is the righteousness of God revealed from faith to faith: as it is written, The just shall live by faith.",1.000000
12,KJVCP,KJV Cambridge Paragraph,"For therein is the righteousness of God revealed from faith to faith: as it is written, The just shall live by faith.",1.000000
10,KJAV,King James Version,"For therein is the righteousness of God revealed from faith to faith: as it is written, The just shall live by faith.",1.000000
16,NWB,Webster Bible,"For in this is the righteousness of God revealed from faith to faith: as it is written, The just shall live by faith.",0.988845
6,EMTV,English Majority Text Version,"For in it the righteousness of God is revealed from faith to faith; as it is written, “The just shall live by faith.”",0.986585
0,ASV,American Standard Version (1901),"For therein is revealed a righteousness of God from faith unto faith: as it is written, But the righteous shall live by faith.",0.972933
1,ASVBT,ASV Byzantine Text,"For therein is revealed a righteousness of God from faith unto faith: as it is written, But the righteous shall live by faith.",0.972933
8,GNB,Noyes Bible,"For therein is revealed the righteousness which is of God from faith to faith; as it is written, “But the righteous shall live by faith.”",0.971429
18,RVA,Revised Version,"For therein is revealed a righteousness of God by faith unto faith: as it is written, But the righteous shall live by faith.",0.967390
29,WEBBE,World English Bible British Edition with Deuterocanon,"For in it is revealed God’s righteousness from faith to faith. As it is written, “But the righteous shall live by faith.”",0.967063


In [6]:
import chromadb
import numpy as np

# Connect to the persisted ChromaDB
client = chromadb.PersistentClient(path="data/verse_embedding")
collection = client.get_collection("bible_verses")

# Fetch all Romans 1:17 verses across translations
results = collection.get(
    where={"$and": [
        {"commonName": {"$eq": "Romans"}},
        {"chapterNumber": {"$eq": 1}},
        {"number": {"$eq": 17}},
    ]},
    include=["embeddings", "documents", "metadatas"],
)

# Build a DataFrame from the results
romans_df = pd.DataFrame({
    "id": results["ids"],
    "text": results["documents"],
    "translationId": [m["translationId"] for m in results["metadatas"]],
    "embedding": list(results["embeddings"]),
})

# Merge with translation names from SQLite
translation_names = pd.read_sql_query(
    "SELECT id, shortName, englishName FROM Translation", conn
)
romans_df = romans_df.merge(
    translation_names, left_on="translationId", right_on="id", suffixes=("", "_t")
)

# Get the KJAV embedding
kjav_row = romans_df[romans_df["shortName"] == "KJAV"].iloc[0]
kjav_emb = np.array(kjav_row["embedding"])

# Compute cosine similarity between KJAV and all others
def cosine_sim(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

romans_df["similarity_to_KJAV"] = romans_df["embedding"].apply(
    lambda emb: cosine_sim(kjav_emb, np.array(emb))
)

# Display sorted results
romans_df[["shortName", "englishName", "text", "similarity_to_KJAV"]].sort_values(
    "similarity_to_KJAV", ascending=False
).reset_index(drop=True)

,shortName,englishName,text,similarity_to_KJAV
0,KJAV,King James Version,"For therein is the righteousness of God revealed from faith to faith: as it is written, The just shall live by faith.",1.000000
1,KJVCP,KJV Cambridge Paragraph,"For therein is the righteousness of God revealed from faith to faith: as it is written, The just shall live by faith.",1.000000
2,KJVA,King James Version + Apocrypha,"For therein is the righteousness of God revealed from faith to faith: as it is written, The just shall live by faith.",1.000000
3,NWB,Webster Bible,"For in this is the righteousness of God revealed from faith to faith: as it is written, The just shall live by faith.",0.988845
4,EMTV,English Majority Text Version,"For in it the righteousness of God is revealed from faith to faith; as it is written, “The just shall live by faith.”",0.986585
5,ASV,American Standard Version (1901),"For therein is revealed a righteousness of God from faith unto faith: as it is written, But the righteous shall live by faith.",0.972933
6,ASVBT,ASV Byzantine Text,"For therein is revealed a righteousness of God from faith unto faith: as it is written, But the righteous shall live by faith.",0.972933
7,GNB,Noyes Bible,"For therein is revealed the righteousness which is of God from faith to faith; as it is written, “But the righteous shall live by faith.”",0.971429
8,RVA,Revised Version,"For therein is revealed a righteousness of God by faith unto faith: as it is written, But the righteous shall live by faith.",0.967390
9,WMB,World Messianic Bible,"For in it is revealed God’s righteousness from faith to faith. As it is written, “But the righteous shall live by faith.”",0.967063


In [7]:
# Fetch all Romans verses for KJAV and BSB
kjav_id = translation_names[translation_names["shortName"] == "KJAV"]["id"].iloc[0]
bsb_id = translation_names[translation_names["shortName"] == "BSB"]["id"].iloc[0]

kjav_results = collection.get(
    where={"$and": [
        {"commonName": {"$eq": "Romans"}},
        {"translationId": {"$eq": kjav_id}},
    ]},
    include=["embeddings", "metadatas"],
)

bsb_results = collection.get(
    where={"$and": [
        {"commonName": {"$eq": "Romans"}},
        {"translationId": {"$eq": bsb_id}},
    ]},
    include=["embeddings", "metadatas"],
)

# Build DataFrames with chapter and embedding
def build_chapter_df(results):
    return pd.DataFrame({
        "chapterNumber": [m["chapterNumber"] for m in results["metadatas"]],
        "embedding": list(results["embeddings"]),
    })

kjav_df = build_chapter_df(kjav_results)
bsb_df = build_chapter_df(bsb_results)

# Max-pool embeddings per chapter
def max_pool_by_chapter(df):
    pooled = {}
    for chapter, group in df.groupby("chapterNumber"):
        embs = np.stack(group["embedding"].values)
        pooled[chapter] = embs.max(axis=0)  # max pooling across verses
    return pooled

kjav_pooled = max_pool_by_chapter(kjav_df)
bsb_pooled = max_pool_by_chapter(bsb_df)

# Compute cosine similarity per chapter
chapters = sorted(set(kjav_pooled.keys()) & set(bsb_pooled.keys()))
similarities = []
for ch in chapters:
    sim = cosine_sim(kjav_pooled[ch], bsb_pooled[ch])
    similarities.append({"chapter": ch, "similarity_KJAV_vs_BSB": sim})

chapter_sim_df = pd.DataFrame(similarities)
chapter_sim_df

,chapter,similarity_KJAV_vs_BSB
0,1,0.972617
1,2,0.968667
2,3,0.975778
3,4,0.965847
4,5,0.970227
5,6,0.973316
6,7,0.968638
7,8,0.982847
8,9,0.974625
9,10,0.973946


In [8]:
# Verse-by-verse comparison of KJAV vs BSB for Romans Chapter 13
kjav_ch13 = collection.get(
    where={"$and": [
        {"commonName": {"$eq": "Romans"}},
        {"chapterNumber": {"$eq": 13}},
        {"translationId": {"$eq": kjav_id}},
    ]},
    include=["embeddings", "documents", "metadatas"],
)

bsb_ch13 = collection.get(
    where={"$and": [
        {"commonName": {"$eq": "Romans"}},
        {"chapterNumber": {"$eq": 13}},
        {"translationId": {"$eq": bsb_id}},
    ]},
    include=["embeddings", "documents", "metadatas"],
)

# Build DataFrames keyed by verse number
kjav_verses = pd.DataFrame({
    "verse": [m["number"] for m in kjav_ch13["metadatas"]],
    "kjav_text": kjav_ch13["documents"],
    "kjav_embedding": list(kjav_ch13["embeddings"]),
}).set_index("verse")

bsb_verses = pd.DataFrame({
    "verse": [m["number"] for m in bsb_ch13["metadatas"]],
    "bsb_text": bsb_ch13["documents"],
    "bsb_embedding": list(bsb_ch13["embeddings"]),
}).set_index("verse")

# Join on verse number and compute similarity
ch13_df = kjav_verses.join(bsb_verses).sort_index()
ch13_df["similarity"] = ch13_df.apply(
    lambda row: cosine_sim(np.array(row["kjav_embedding"]), np.array(row["bsb_embedding"])),
    axis=1,
)

ch13_df[["kjav_text", "bsb_text", "similarity"]]

,kjav_text,bsb_text,similarity
verse,,,
1,Let every soul be subject unto the higher powers. For there is no power but of God: the powers that be are ordained of God.,"Everyone must submit himself to the governing authorities, for there is no authority except that which is from God. The authorities that exist have been appointed by God.",0.818539
2,"Whosoever therefore resisteth the power, resisteth the ordinance of God: and they that resist shall receive to themselves damnation.","Consequently, whoever resists authority is opposing what God has set in place, and those who do so will bring judgment on themselves.",0.818528
3,"For rulers are not a terror to good works, but to the evil. Wilt thou then not be afraid of the power? do that which is good, and thou shalt have praise of the same:","For rulers are not a terror to good conduct, but to bad. Do you want to be unafraid of the one in authority? Then do what is right, and you will have his approval.",0.854235
4,"For he is the minister of God to thee for good. But if thou do that which is evil, be afraid; for he beareth not the sword in vain: for he is the minister of God, a revenger to execute wrath upon him that doeth evil.","For he is God’s servant for your good. But if you do wrong, be afraid, for he does not carry the sword in vain. He is God’s servant, an agent of retribution to the wrongdoer.",0.865408
5,"Wherefore ye must needs be subject, not only for wrath, but also for conscience sake.","Therefore it is necessary to submit to authority, not only to avoid punishment, but also as a matter of conscience.",0.723573
6,"For for this cause pay ye tribute also: for they are God’s ministers, attending continually upon this very thing.","This is also why you pay taxes. For the authorities are God’s servants, who devote themselves to their work.",0.711164
7,Render therefore to all their dues: tribute to whom tribute is due; custom to whom custom; fear to whom fear; honour to whom honour.,"Pay everyone what you owe him: taxes to whom taxes are due, revenue to whom revenue is due, respect to whom respect is due, honor to whom honor is due.",0.758109
8,"Owe no man any thing, but to love one another: for he that loveth another hath fulfilled the law.","Be indebted to no one, except to one another in love. For he who loves his neighbor has fulfilled the law.",0.879577
9,"For this, Thou shalt not commit adultery, Thou shalt not kill, Thou shalt not steal, Thou shalt not bear false witness, Thou shalt not covet; and if there be any other commandment, it is briefly comprehended in this saying, namely, Thou shalt love thy neighbour as thyself.","The commandments “Do not commit adultery,” “Do not murder,” “Do not steal,” “Do not covet,” and any other commandments, are summed up in this one decree: “Love your neighbor as yourself.”",0.867041


In [9]:
# Full Bible comparison: KJAV vs BSB using max pooling per book
from tqdm.notebook import tqdm

# Fetch ALL verses for KJAV and BSB from ChromaDB
kjav_all = collection.get(
    where={"translationId": {"$eq": kjav_id}},
    include=["embeddings", "metadatas"],
)

bsb_all = collection.get(
    where={"translationId": {"$eq": bsb_id}},
    include=["embeddings", "metadatas"],
)

def build_book_df(results):
    return pd.DataFrame({
        "commonName": [m["commonName"] for m in results["metadatas"]],
        "embedding": list(results["embeddings"]),
    })

kjav_book_df = build_book_df(kjav_all)
bsb_book_df = build_book_df(bsb_all)

# Max-pool embeddings per book
def max_pool_by_book(df):
    pooled = {}
    for book, group in df.groupby("commonName"):
        embs = np.stack(group["embedding"].values)
        pooled[book] = embs.max(axis=0)
    return pooled

kjav_book_pooled = max_pool_by_book(kjav_book_df)
bsb_book_pooled = max_pool_by_book(bsb_book_df)

# Compute cosine similarity per book (in canonical order from SQLite)
book_order = pd.read_sql_query(
    f'SELECT DISTINCT b.commonName, b."order" FROM Book b WHERE b.translationId = "{kjav_id}" ORDER BY b."order"',
    conn,
)

book_similarities = []
for _, row in book_order.iterrows():
    book = row["commonName"]
    if book in kjav_book_pooled and book in bsb_book_pooled:
        sim = cosine_sim(kjav_book_pooled[book], bsb_book_pooled[book])
        book_similarities.append({"book": book, "similarity_KJAV_vs_BSB": sim})

bible_sim_df = pd.DataFrame(book_similarities)

# Also compute the whole-Bible max-pooled similarity
kjav_bible_emb = np.stack(list(kjav_book_pooled.values())).max(axis=0)
bsb_bible_emb = np.stack(list(bsb_book_pooled.values())).max(axis=0)
whole_bible_sim = cosine_sim(kjav_bible_emb, bsb_bible_emb)

print(f"Whole-Bible max-pooled similarity (KJAV vs BSB): {whole_bible_sim:.6f}\n")
print(f"Book-by-book comparison ({len(bible_sim_df)} books):")
bible_sim_df

Whole-Bible max-pooled similarity (KJAV vs BSB): 0.995133

Book-by-book comparison (66 books):


,book,similarity_KJAV_vs_BSB
0,Genesis,0.992556
1,Exodus,0.991941
2,Leviticus,0.986820
3,Numbers,0.991640
4,Deuteronomy,0.989122
...,...,...
61,1 John,0.983560
62,2 John,0.959315
63,3 John,0.959102
64,Jude,0.971825


In [10]:
# Find the book with the biggest difference
least_similar = bible_sim_df.loc[bible_sim_df["similarity_KJAV_vs_BSB"].idxmin()]
book_name = least_similar["book"]
print(f"Most divergent book: {book_name} (similarity: {least_similar['similarity_KJAV_vs_BSB']:.6f})\n")

# Fetch all verses for that book from both translations (already in kjav_all / bsb_all)
kjav_book = pd.DataFrame({
    "commonName": [m["commonName"] for m in kjav_all["metadatas"]],
    "chapterNumber": [m["chapterNumber"] for m in kjav_all["metadatas"]],
    "embedding": list(kjav_all["embeddings"]),
})
kjav_book = kjav_book[kjav_book["commonName"] == book_name]

bsb_book = pd.DataFrame({
    "commonName": [m["commonName"] for m in bsb_all["metadatas"]],
    "chapterNumber": [m["chapterNumber"] for m in bsb_all["metadatas"]],
    "embedding": list(bsb_all["embeddings"]),
})
bsb_book = bsb_book[bsb_book["commonName"] == book_name]

# Max-pool per chapter
kjav_ch_pooled = max_pool_by_chapter(kjav_book)
bsb_ch_pooled = max_pool_by_chapter(bsb_book)

# Compare chapter by chapter
ch_list = sorted(set(kjav_ch_pooled.keys()) & set(bsb_ch_pooled.keys()))
ch_sims = []
for ch in ch_list:
    s = cosine_sim(kjav_ch_pooled[ch], bsb_ch_pooled[ch])
    ch_sims.append({"chapter": ch, "similarity_KJAV_vs_BSB": s})

book_ch_df = pd.DataFrame(ch_sims)
print(f"Chapter-by-chapter comparison for {book_name}:")
book_ch_df

Most divergent book: 3 John (similarity: 0.959102)

Chapter-by-chapter comparison for 3 John:


,chapter,similarity_KJAV_vs_BSB
0,1,0.959102


In [11]:
# Verse-by-verse comparison of KJAV vs BSB for 3 John Chapter 1
kjav_3john = collection.get(
    where={"$and": [
        {"commonName": {"$eq": "3 John"}},
        {"chapterNumber": {"$eq": 1}},
        {"translationId": {"$eq": kjav_id}},
    ]},
    include=["embeddings", "documents", "metadatas"],
)

bsb_3john = collection.get(
    where={"$and": [
        {"commonName": {"$eq": "3 John"}},
        {"chapterNumber": {"$eq": 1}},
        {"translationId": {"$eq": bsb_id}},
    ]},
    include=["embeddings", "documents", "metadatas"],
)

kjav_3j_verses = pd.DataFrame({
    "verse": [m["number"] for m in kjav_3john["metadatas"]],
    "kjav_text": kjav_3john["documents"],
    "kjav_embedding": list(kjav_3john["embeddings"]),
}).set_index("verse")

bsb_3j_verses = pd.DataFrame({
    "verse": [m["number"] for m in bsb_3john["metadatas"]],
    "bsb_text": bsb_3john["documents"],
    "bsb_embedding": list(bsb_3john["embeddings"]),
}).set_index("verse")

john3_df = kjav_3j_verses.join(bsb_3j_verses).sort_index()
john3_df["similarity"] = john3_df.apply(
    lambda row: cosine_sim(np.array(row["kjav_embedding"]), np.array(row["bsb_embedding"])),
    axis=1,
)

john3_df[["kjav_text", "bsb_text", "similarity"]]

,kjav_text,bsb_text,similarity
verse,,,
1,"The elder unto the wellbeloved Gaius, whom I love in the truth.","The elder, \nTo the beloved Gaius, whom I love in the truth:",0.934122
2,"Beloved, I wish above all things that thou mayest prosper and be in health, even as thy soul prospereth.","Beloved, I pray that in every way you may prosper and enjoy good health, as your soul also prospers.",0.921357
3,"For I rejoiced greatly, when the brethren came and testified of the truth that is in thee, even as thou walkest in the truth.","For I was overjoyed when the brothers came and testified about your devotion to the truth, in which you continue to walk.",0.889167
4,I have no greater joy than to hear that my children walk in truth.,I have no greater joy than to hear that my children are walking in the truth.,0.973239
5,"Beloved, thou doest faithfully whatsoever thou doest to the brethren, and to strangers;","Beloved, you are faithful in what you are doing for the brothers, and especially since they are strangers to you.",0.860814
6,"Which have borne witness of thy charity before the church: whom if thou bring forward on their journey after a godly sort, thou shalt do well:",They have testified to the church about your love. You will do well to send them on their way in a manner worthy of God.,0.755671
7,"Because that for his name’s sake they went forth, taking nothing of the Gentiles.","For they went out on behalf of the Name, accepting nothing from the Gentiles.",0.841859
8,"We therefore ought to receive such, that we might be fellowhelpers to the truth.","Therefore we ought to support such men, so that we may be fellow workers for the truth.",0.765418
9,"I wrote unto the church: but Diotrephes, who loveth to have the preeminence among them, receiveth us not.","I have written to the church about this, but Diotrephes, who loves to be first, will not accept our instruction.",0.857315


In [12]:
# All-translations vs KJAV similarity (max-pooled per book, shared books only)
# Fetch ALL embeddings from ChromaDB in batches to avoid SQL variable limits
from tqdm.notebook import tqdm

total_count = collection.count()
print(f"Total verses in ChromaDB: {total_count:,}")

FETCH_BATCH = 10_000
all_metadatas = []
all_embeddings = []

for offset in tqdm(range(0, total_count, FETCH_BATCH), desc="Fetching"):
    batch = collection.get(
        limit=FETCH_BATCH,
        offset=offset,
        include=["embeddings", "metadatas"],
    )
    all_metadatas.extend(batch["metadatas"])
    all_embeddings.extend(batch["embeddings"])

print(f"Fetched {len(all_metadatas):,} verses.\n")

# Build a master DataFrame
master_df = pd.DataFrame({
    "translationId": [m["translationId"] for m in all_metadatas],
    "commonName": [m["commonName"] for m in all_metadatas],
    "embedding": all_embeddings,
})

# Max-pool per (translation, book)
print("Max-pooling per (translation, book)...")
pooled_by_trans = {}
for (t_id, book), group in tqdm(master_df.groupby(["translationId", "commonName"]), desc="Pooling"):
    embs = np.stack(group["embedding"].values)
    pooled_by_trans.setdefault(t_id, {})[book] = embs.max(axis=0)

# KJAV reference
kjav_books = set(pooled_by_trans[kjav_id].keys())
print(f"\nKJAV has {len(kjav_books)} books. Comparing against {len(pooled_by_trans) - 1} translations...\n")

results_list = []
for t_id, book_pooled in pooled_by_trans.items():
    if t_id == kjav_id:
        continue
    
    t_info = translation_names[translation_names["id"] == t_id].iloc[0]
    t_books = set(book_pooled.keys())
    
    shared = kjav_books & t_books
    extra_in_t = t_books - kjav_books
    missing_from_t = kjav_books - t_books
    
    if not shared:
        continue
    
    kjav_shared_emb = np.stack([pooled_by_trans[kjav_id][b] for b in shared]).max(axis=0)
    t_shared_emb = np.stack([book_pooled[b] for b in shared]).max(axis=0)
    
    whole_sim = cosine_sim(kjav_shared_emb, t_shared_emb)
    
    results_list.append({
        "translation": t_info["shortName"],
        "translation_name": t_info["englishName"],
        "similarity_to_KJAV": whole_sim,
        "shared_books": len(shared),
        "extra_books": len(extra_in_t),
        "missing_books": len(missing_from_t),
        "extra_book_names": ", ".join(sorted(extra_in_t)) if extra_in_t else "",
        "missing_book_names": ", ".join(sorted(missing_from_t)) if missing_from_t else "",
    })

all_sim_df = pd.DataFrame(results_list).sort_values("similarity_to_KJAV", ascending=False).reset_index(drop=True)

print(f"All translations ranked by similarity to KJAV:")
print(f"(Only shared books compared; 'extra' = Apocrypha/additions in that translation)\n")
all_sim_df

Total verses in ChromaDB: 1,250,711


Fetching:   0%|          | 0/126 [00:00<?, ?it/s]

Fetched 1,250,711 verses.

Max-pooling per (translation, book)...


Pooling:   0%|          | 0/2667 [00:00<?, ?it/s]


KJAV has 66 books. Comparing against 48 translations...

All translations ranked by similarity to KJAV:
(Only shared books compared; 'extra' = Apocrypha/additions in that translation)



,translation,translation_name,similarity_to_KJAV,shared_books,extra_books,missing_books,extra_book_names,missing_book_names
0,KJVA,King James Version + Apocrypha,1.000000,66,14,0,"1 Esdras, 1 Maccabees, 2 Esdras, 2 Maccabees, Azariah, Baruch, Bel, Esther (Greek), Judith, Manasseh, Sirach, Susanna, Tobit, Wisdom",
1,KJVCP,KJV Cambridge Paragraph,0.999432,66,14,0,"1 Esdras, 1 Maccabees, 2 Esdras, 2 Maccabees, Baruch, Bel and the Dragon, Ecclesiasticus, Judith, Prayer of Manasses, Song of the Three Holy Children, Susanna, The Rest of Esther, Tobit, Wisdom of Solomon",
2,NWB,Webster Bible,0.999332,66,0,0,,
3,ASVBT,ASV Byzantine Text,0.998137,66,18,0,"1 Esdras, 1 Maccabees, 2 Esdras, 2 Maccabees, 3 Maccabees, 4 Maccabees, Additions to Esther, Baruch, Bel and the Dragon, Ecclesiasticus, Judith, Prayer of Manasseh, Psalm 151, Psalms of Solomon, Song of the Three Holy Children, Susanna, The Wisdom Of Solomon, Tobit",
4,RVA,Revised Version,0.997919,66,14,0,"1 Esdras, 2 Esdras, 3 Holy Children’s Song, Additions to Esther, Baruch, Bel and the Dragon, Ecclesiasticus, I Maccabees, II Maccabees, Judith, Prayer of Manasses, Susanna, The Wisdom Of Solomon, Tobit",
5,JPSTN,JPS TaNaKH 1917,0.997843,39,0,27,,"1 Corinthians, 1 John, 1 Peter, 1 Thessalonians, 1 Timothy, 2 Corinthians, 2 John, 2 Peter, 2 Thessalonians, 2 Timothy, 3 John, Acts, Colossians, Ephesians, Galatians, Hebrews, James, John, Jude, Luke, Mark, Matthew, Philemon, Philippians, Revelation, Romans, Titus"
6,ASV,American Standard Version (1901),0.997566,66,0,0,,
7,DBY,Darby Translation,0.997458,66,0,0,,
8,WEBBE,World English Bible British Edition,0.996523,66,0,0,,
9,WEBBE,World English Bible British Edition with Deuterocanon,0.996523,66,15,0,"1 Esdras, 1 Maccabees, 2 Esdras, 2 Maccabees, 3 Maccabees, 4 Maccabees, Baruch, Daniel (Greek), Esther (Greek), Judith, Prayer of Manasses, Psalm 151, Sirach, Tobit, Wisdom of Solomon",


In [13]:
# 2D visualization of Romans max-pooled embeddings per translation using t-SNE
from sklearn.manifold import TSNE
import plotly.express as px

# Filter master_df to Romans only
romans_master = master_df[master_df["commonName"] == "Romans"]

# Max-pool all Romans verses into a single embedding per translation
romans_pooled = {}
for t_id, group in romans_master.groupby("translationId"):
    embs = np.stack(group["embedding"].values)
    romans_pooled[t_id] = embs.max(axis=0)

# Build matrix and labels
t_ids = list(romans_pooled.keys())
embeddings_matrix = np.stack([romans_pooled[t] for t in t_ids])

short_labels = []
full_labels = []
for t in t_ids:
    match = translation_names[translation_names["id"] == t]
    if not match.empty:
        short_labels.append(match.iloc[0]["shortName"])
        full_labels.append(match.iloc[0]["englishName"])
    else:
        short_labels.append(t)
        full_labels.append(t)

# t-SNE to 2D
tsne = TSNE(n_components=2, perplexity=min(15, len(t_ids) - 1), random_state=42, metric="cosine")
coords_2d = tsne.fit_transform(embeddings_matrix)

# Build plot DataFrame
plot_df = pd.DataFrame({
    "x": coords_2d[:, 0],
    "y": coords_2d[:, 1],
    "translation": short_labels,
    "translation_name": full_labels,
})

fig = px.scatter(
    plot_df, x="x", y="y", text="translation",
    hover_data={"translation_name": True, "translation": True, "x": False, "y": False},
    title="Romans: Translation Clustering (t-SNE of max-pooled embeddings)",
    template="simple_white",
    width=900, height=700,  
)
fig.update_traces(textposition="top center", marker=dict(size=8))
fig.update_layout(
    xaxis_title="t-SNE 1", yaxis_title="t-SNE 2",
    showlegend=False,
) 
fig.show()

In [14]:
# 2D visualization of whole-Bible max-pooled embeddings per translation using t-SNE

# Max-pool ALL verses into a single embedding per translation
bible_pooled = {}
for t_id, group in master_df.groupby("translationId"):
    embs = np.stack(group["embedding"].values)
    bible_pooled[t_id] = embs.max(axis=0)

# Build matrix and labels
t_ids_all = list(bible_pooled.keys())
bible_emb_matrix = np.stack([bible_pooled[t] for t in t_ids_all])

short_labels_all = []
full_labels_all = []
for t in t_ids_all:
    match = translation_names[translation_names["id"] == t]
    if not match.empty:
        short_labels_all.append(match.iloc[0]["shortName"])
        full_labels_all.append(match.iloc[0]["englishName"])
    else:
        short_labels_all.append(t)
        full_labels_all.append(t)

# t-SNE to 2D
tsne_bible = TSNE(n_components=2, perplexity=min(15, len(t_ids_all) - 1), random_state=42, metric="cosine")
coords_bible_2d = tsne_bible.fit_transform(bible_emb_matrix)

# Build plot DataFrame
plot_bible_df = pd.DataFrame({
    "x": coords_bible_2d[:, 0],
    "y": coords_bible_2d[:, 1],
    "translation": short_labels_all,
    "translation_name": full_labels_all,
})

fig_bible = px.scatter(
    plot_bible_df, x="x", y="y", text="translation",
    hover_data={"translation_name": True, "translation": True, "x": False, "y": False},
    title="Whole Bible: Translation Clustering (t-SNE of max-pooled embeddings)",
    template="simple_white",
    width=900, height=700,
)
fig_bible.update_traces(textposition="top center", marker=dict(size=8))
fig_bible.update_layout(
    xaxis_title="t-SNE 1", yaxis_title="t-SNE 2",
    showlegend=False,
)
fig_bible.show()